# Food generator expansion verification

Timestamp: 2026-09-11 23:14 +0400

## Hypothesis / intent and method

The dependency-free page should expose exactly 20 uniquely named dishes, map every forced first pick to the exact label and photo, and allow no consecutive dish repeat. The inline JavaScript was parsed and executed through macOS JXA/osascript with a mocked DOM. Fresh executions forced all 20 first-pick intervals; persistent executions made 120 constant-random and 120 varied deterministic clicks. All 20 photo URLs were extracted and requested with curl. Paths and both notebooks were checked directly.

In [1]:
// Core snippets actually run with: osascript -l JavaScript -e '...'
const rows = [...source.matchAll(/name:\\s*\"([^\"]+)\"\\s*,\\s*photo:\\s*\"([^\"]+)\"/g)].map(m => ({name:m[1], photo:m[2]}));
rows.forEach((row, i) => { const nodes=fresh([(i+0.25)/20]); nodes["#generate-button"].click(); assert(nodes["#result"].textContent === `Today's pick: ${row.name}! 🥳`); assert(nodes["#food-photo"].src === row.photo); });
sequential("constant", [0], 120); sequential("varied", [0,0.11,0.99,0.43,0.76,0.25,0.62], 120);

PASS: 20 unique options; 20 forced first-pick labels/photos; 240 sequential clicks (120 constant + 120 varied) with no consecutive repeats.


In [2]:
# Core Python/curl snippet actually run:
urls = list(dict.fromkeys(re.findall(r"https://images\\.unsplash\\.com/[^\"]+", html)))
p = subprocess.run(["curl", "-L", "--fail", "--silent", "--show-error", "--output", "/dev/null", "--max-time", "30", "--write-out", "%{http_code}", url], text=True, capture_output=True)
assert len(urls) == 20 and p.returncode == 0

PASS: all 20 photo URLs returned final HTTP 200 via curl; Falafel Bowl substitute photo ID 1547058881-aa0edd92aab3 returned HTTP 200.


In [3]:
python3 -m json.tool food_generator/experiments/2026-09-11-2154-lunch-picker.ipynb >/dev/null
python3 -m json.tool food_generator/experiments/2026-09-11-2314-food-generator-expansion.ipynb >/dev/null
test ! -e index.html && test ! -e 2026-09-11-2154-lunch-picker.ipynb
test -f food_generator/index.html && test -f food_generator/experiments/2026-09-11-2154-lunch-picker.ipynb && test -f food_generator/experiments/2026-09-11-2314-food-generator-expansion.ipynb

PASS: both notebooks are valid JSON; root copies absent; destination files exist.


## Result interpretation

All requested application checks passed. There are 20 unique names; exact name/photo mappings passed for every first-pick interval; 240 sequential deterministic clicks had no consecutive repeat; and all 20 photo URLs returned HTTP 200. Falafel Bowl uses reachable substitute photo ID `1547058881-aa0edd92aab3`, differing from the originally supplied candidate. Root copies are absent and the app remains at `food_generator/index.html`.

No application check failed. Two ancillary tool probes failed without changing files: `git rev-parse --short HEAD` exited 128 because the repository has no commits, and `jq empty ...` exited 127 because jq is not installed. Both notebooks were instead parsed successfully with `python3 -m json.tool`.